# DCC-GARCH Allocation with Trading Costs

Tests close-based covariance estimates next-session and tracks allocation turnover and trading costs.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
============================================================
Production-Ready DCC-GARCH(1,1) Trading Strategy
Magnificent Seven (MAG7) Portfolio Optimization
============================================================

Key Features:
- Proper trading flow: Signal at Close[t] → Execute at Open[t+1]
- No look-ahead bias: Uses only T-1 data for T decisions
- Real execution costs and slippage
- Yahoo Finance data integration
- Benchmark comparison with $URTH
- Open-to-Open returns calculation

Requirements:
pip install pandas numpy scipy arch matplotlib
============================================================
"""

from _market_data import download_price_records
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from arch import arch_model
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

MAG7_TICKERS = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "TSLA"]
BENCHMARK_TICKER = "SPY"
BACKTEST_START = "2019-01-01"
BACKTEST_END = "2025-12-31"
GARCH_P = 1
GARCH_Q = 1
GARCH_DISTRIBUTION = "t"
GARCH_MEAN_MODEL = "Constant"
DCC_INITIAL_A = 0.02
DCC_INITIAL_B = 0.97
DCC_MAX_AB_SUM = 0.999
TRANSACTION_COST_BPS = 5.0
SLIPPAGE_BPS = 2.0
REBALANCE_FREQUENCY = "weekly"
ANNUALIZATION_FACTOR = 252
RISK_FREE_RATE = 0.0
MIN_VOLATILITY = 1e-08
REGULARIZATION_LAMBDA = 1e-10


def fetch_fmp_historical_prices(ticker, start_date, end_date):
    """
    Fetch historical OHLC data from Yahoo Finance for a single ticker.

    IMPORTANT: This function retrieves actual market data without any forward-looking bias.
    Data is fetched with proper date alignment to ensure no future information leakage.

    Parameters:
    -----------
    ticker : str
        Stock ticker symbol
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str
        End date in 'YYYY-MM-DD' format
        Yahoo Finance key

    Returns:
    --------
    pd.DataFrame
        DataFrame with Open, High, Low, Close prices indexed by date

    Notes:
    ------
    - No bfill/ffill used - missing data remains as NaN
    - Date is set as index for proper time-series operations
    - All prices are adjusted for splits/dividends via Yahoo Finance's adjusted prices
    """
    try:
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data or len(data["historical"]) == 0:
            print(f"Warning: No data returned for {ticker}")
            return pd.DataFrame()
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()
        df = df[["open", "high", "low", "close"]].rename(
            columns={"open": "Open", "high": "High", "low": "Low", "close": "Close"}
        )
        return df
    except RuntimeError as e:
        print(f"Error fetching data for {ticker}: {e}")
        return pd.DataFrame()


def load_portfolio_data(tickers, start_date, end_date):
    """
    Load OHLC data for multiple tickers from Yahoo Finance.

    CRITICAL: No data imputation or forward-filling is performed.
    Only dates where ALL tickers have valid data are retained to prevent
    look-ahead bias and ensure strategy viability.

    Parameters:
    -----------
    tickers : list
        List of ticker symbols
    start_date : str
        Start date
    end_date : str
        End date
        Yahoo Finance key

    Returns:
    --------
    dict
        Dictionary with keys 'Open', 'Close' containing DataFrames
        Each DataFrame has tickers as columns and dates as index
    """
    all_data = {}
    print(f"Fetching data for {len(tickers)} tickers from Yahoo Finance...")
    for ticker in tickers:
        print(f"  Downloading {ticker}...")
        df = fetch_fmp_historical_prices(ticker, start_date, end_date)
        if not df.empty:
            all_data[ticker] = df
    if len(all_data) == 0:
        raise ValueError("No data retrieved for any ticker. Check API key and tickers.")
    open_prices = pd.DataFrame({ticker: data["Open"] for ticker, data in all_data.items()})
    close_prices = pd.DataFrame({ticker: data["Close"] for ticker, data in all_data.items()})
    initial_rows = len(open_prices)
    open_prices = open_prices.dropna()
    close_prices = close_prices.dropna()
    common_dates = open_prices.index.intersection(close_prices.index)
    open_prices = open_prices.loc[common_dates]
    close_prices = close_prices.loc[common_dates]
    dropped_rows = initial_rows - len(open_prices)
    if dropped_rows > 0:
        print(f"Dropped {dropped_rows} rows with missing data (no forward/backward filling)")
    print(f"Final dataset: {len(open_prices)} days, {len(tickers)} tickers")
    return {"Open": open_prices, "Close": close_prices}


def fetch_benchmark_data(ticker, start_date, end_date):
    """
    Fetch benchmark data (e.g., URTH) for performance comparison.

    Returns both Open and Close prices for proper comparison with strategy.
    """
    print(f"Fetching benchmark {ticker}...")
    df = fetch_fmp_historical_prices(ticker, start_date, end_date)
    return df


def calculate_close_to_close_returns(close_prices, freq=ANNUALIZATION_FACTOR):
    """
    Calculate Close-to-Close log returns for signal generation.

    IMPORTANT: These returns are used ONLY for volatility modeling and correlation estimation.
    They are NOT used for P&L calculation.

    Formula: r[t] = ln(Close[t] / Close[t-1])

    This ensures we only use information available at Close[t] to generate signal for t+1.
    """
    returns = np.log(close_prices / close_prices.shift(1))
    return returns.dropna()


def calculate_open_to_open_returns(open_prices):
    """
    Calculate Open-to-Open returns for P&L attribution.

    CRITICAL TRADING FLOW:
    - Signal generated at Close[t]
    - Position entered at Open[t+1]
    - Position valued at Open[t+2]
    - Return = (Open[t+2] / Open[t+1]) - 1

    This matches real trading where you:
    1. Decide weights at market close
    2. Execute next morning at market open
    3. Hold until next open for P&L calculation
    """
    returns = open_prices / open_prices.shift(1) - 1
    return returns.dropna()


def fit_univariate_garch(
    returns_df, p=GARCH_P, q=GARCH_Q, dist=GARCH_DISTRIBUTION, mean=GARCH_MEAN_MODEL
):
    """
    Fit univariate GARCH(1,1) with Student-t innovations for each asset.

    METHODOLOGY:
    - Uses Close-to-Close returns for volatility estimation
    - Student-t distribution captures fat tails in financial returns
    - Returns conditional volatility and standardized residuals

    IMPORTANT: Scaling to percentage returns (×100) improves numerical stability
    in GARCH optimization. We scale back to original units after fitting.

    Parameters:
    -----------
    returns_df : pd.DataFrame
        Close-to-Close log returns

    Returns:
    --------
    models : dict
        Fitted GARCH models for each asset
    cond_vol : pd.DataFrame
        Conditional volatility (sigma_t) for each asset
    z : pd.DataFrame
        Standardized residuals (innovations / sigma_t)
    """
    models = {}
    cond_vol = pd.DataFrame(index=returns_df.index, columns=returns_df.columns, dtype=float)
    z = pd.DataFrame(index=returns_df.index, columns=returns_df.columns, dtype=float)
    print("\nFitting univariate GARCH models...")
    for col in returns_df.columns:
        print(f"  Estimating GARCH for {col}...")
        r = returns_df[col].astype(float) * 100.0
        am = arch_model(r, vol="GARCH", p=p, q=q, dist=dist, mean=mean)
        res = am.fit(disp="off")
        sigma_t = res.conditional_volatility.values / 100.0
        sigma_t = np.maximum(sigma_t, MIN_VOLATILITY)
        cond_vol[col] = sigma_t
        z[col] = returns_df[col].values / sigma_t
        models[col] = res
    return (models, cond_vol, z)


def dcc_loglik(params, z_mat, S):
    """
    Negative log-likelihood for DCC(1,1) model.

    MODEL SPECIFICATION:
    Q[t] = (1-a-b)*S + a*(z[t-1] @ z[t-1].T) + b*Q[t-1]
    R[t] = diag(Q[t])^(-1/2) @ Q[t] @ diag(Q[t])^(-1/2)

    Where:
    - S is the unconditional correlation matrix of standardized residuals
    - Q[t] is the dynamic covariance matrix
    - R[t] is the dynamic correlation matrix
    - a (alpha) captures short-term shock impact
    - b (beta) captures persistence

    STATIONARITY: We require a + b < 1 for mean reversion

    Parameters:
    -----------
    params : array
        [a, b] parameters
    z_mat : ndarray
        Matrix of standardized residuals (T x N)
    S : ndarray
        Unconditional correlation matrix
    """
    a, b = params
    if a < 0 or b < 0 or a + b >= DCC_MAX_AB_SUM:
        penalty = 1000000.0 * (1 + abs(a) + abs(b) + max(0, a + b - DCC_MAX_AB_SUM))
        return penalty
    T, N = z_mat.shape
    Q = S.copy()

    def inv_sqrt_diag(M):
        """Compute D^(-1/2) where D = diag(M)"""
        diag_vals = np.diag(M)
        diag_vals = np.clip(diag_vals, MIN_VOLATILITY, None)
        return np.diag(1.0 / np.sqrt(diag_vals))

    ll = 0.0
    for t in range(1, T):
        eps = z_mat[t - 1][:, None]
        Q = (1 - a - b) * S + a * (eps @ eps.T) + b * Q
        Dinv = inv_sqrt_diag(Q)
        R = Dinv @ Q @ Dinv
        R = (R + R.T) / 2.0
        try:
            sign, logdet = np.linalg.slogdet(R)
            if sign <= 0:
                return 1000000.0
            R_reg = R + REGULARIZATION_LAMBDA * np.eye(N)
            invR = np.linalg.inv(R_reg)
        except np.linalg.LinAlgError:
            return 1000000.0
        zt = z_mat[t][:, None]
        ll += logdet + (zt.T @ invR @ zt).item()
    return ll


def fit_dcc(z_df, a0=DCC_INITIAL_A, b0=DCC_INITIAL_B):
    """
    Estimate DCC parameters (a, b) via Quasi-Maximum Likelihood Estimation.

    OPTIMIZATION:
    - Uses L-BFGS-B for constrained optimization
    - Bounds ensure parameters are positive and sum < 1
    - Initial values typically: a ≈ 0.02, b ≈ 0.97 (high persistence)

    Returns:
    --------
    dict with 'a', 'b', 'success' flag, and optimizer result
    """
    z_mat = z_df.values.astype(float)
    S = np.corrcoef(z_mat.T)
    print(f"\nEstimating DCC parameters (initial: a={a0}, b={b0})...")
    x0 = np.array([a0, b0])
    bounds = [(1e-06, 0.499), (1e-06, 0.499)]
    res = minimize(
        dcc_loglik, x0, args=(z_mat, S), method="L-BFGS-B", bounds=bounds, options={"maxiter": 500}
    )
    a, b = res.x
    print(f"  DCC estimation {('converged' if res.success else 'did not converge')}")
    print(f"  Final parameters: a={a:.6f}, b={b:.6f}, a+b={a + b:.6f}")
    return {"a": float(a), "b": float(b), "success": bool(res.success), "opt": res}


def dcc_filter(z_df, cond_vol_df, a, b):
    """
    Reconstruct time-varying correlation R[t] and covariance Σ[t] matrices.

    METHODOLOGY:
    For each time t:
    1. Update Q[t] via DCC recursion
    2. Compute correlation R[t] = D_Q^(-1/2) Q[t] D_Q^(-1/2)
    3. Reconstruct covariance Σ[t] = D_σ[t] R[t] D_σ[t]

    Where D_σ[t] = diag(σ_1[t], ..., σ_N[t]) from GARCH models

    Parameters:
    -----------
    z_df : pd.DataFrame
        Standardized residuals
    cond_vol_df : pd.DataFrame
        Conditional volatilities from GARCH
    a, b : float
        DCC parameters

    Returns:
    --------
    R_t : dict
        Time-varying correlation matrices {date: correlation_matrix}
    Sigma_t : dict
        Time-varying covariance matrices {date: covariance_matrix}
    """
    idx = z_df.index
    cols = z_df.columns
    z_mat = z_df.values
    T, N = z_mat.shape
    S = np.corrcoef(z_mat.T)
    Q = S.copy()

    def inv_sqrt_diag(M):
        diag_vals = np.clip(np.diag(M), MIN_VOLATILITY, None)
        return np.diag(1.0 / np.sqrt(diag_vals))

    R_list = []
    Sigma_list = []
    print("\nFiltering DCC dynamics...")
    for t in range(T):
        if t > 0:
            eps = z_mat[t - 1][:, None]
            Q = (1 - a - b) * S + a * (eps @ eps.T) + b * Q
        Dinv = inv_sqrt_diag(Q)
        R = Dinv @ Q @ Dinv
        R = (R + R.T) / 2.0
        Dt = np.diag(cond_vol_df.iloc[t].values)
        Sigma_t = Dt @ R @ Dt
        R_list.append(R)
        Sigma_list.append(Sigma_t)
    R_t = {idx[t]: pd.DataFrame(R_list[t], index=cols, columns=cols) for t in range(T)}
    Sigma_t = {idx[t]: pd.DataFrame(Sigma_list[t], index=cols, columns=cols) for t in range(T)}
    print(f"  Generated {len(R_t)} time-varying covariance matrices")
    return (R_t, Sigma_t)


def min_var_weights(Sigma_t_dict):
    """
    Compute Global Minimum Variance Portfolio weights for each time period.

    ANALYTICAL SOLUTION:
    w* = Σ^(-1) 1 / (1' Σ^(-1) 1)

    Where:
    - Σ is the forecasted covariance matrix at time t
    - 1 is a vector of ones (sum-to-one constraint)
    - This gives the portfolio with minimum variance (risk)

    IMPORTANT: These are the TARGET weights to be executed at next open.

    NO CONSTRAINTS:
    - Long-only: Can be negative (short positions)
    - No leverage limit in base version
    - Can add constraints if needed for production

    Parameters:
    -----------
    Sigma_t_dict : dict
        Dictionary of covariance matrices by date

    Returns:
    --------
    pd.DataFrame
        Weights for each asset at each rebalancing date
    """
    dates = list(Sigma_t_dict.keys())
    cols = Sigma_t_dict[dates[0]].columns
    W = pd.DataFrame(index=dates, columns=cols, dtype=float)
    print("\nComputing minimum variance weights...")
    for dt in dates:
        S = Sigma_t_dict[dt].values
        N = S.shape[0]
        iota = np.ones((N, 1))
        try:
            Sinv = np.linalg.inv(S + REGULARIZATION_LAMBDA * np.eye(N))
        except np.linalg.LinAlgError:
            Sinv = np.linalg.pinv(S + MIN_VOLATILITY * np.eye(N))
        num = Sinv @ iota
        den = (iota.T @ Sinv @ iota).item()
        w = (num / den).ravel()
        W.loc[dt] = w
    print(f"  Computed weights for {len(W)} rebalancing dates")
    return W


def calculate_turnover(weights_df):
    """
    Calculate portfolio turnover as sum of absolute weight changes.

    Turnover[t] = Σ |w[t] - w[t-1]|

    This measures trading activity and is used to compute transaction costs.
    """
    turnover = weights_df.diff().abs().sum(axis=1)
    return turnover


def backtest_strategy(
    weights_df,
    open_returns_df,
    close_prices_df,
    open_prices_df,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    slippage_bps=SLIPPAGE_BPS,
    freq=ANNUALIZATION_FACTOR,
    rf=RISK_FREE_RATE,
):
    """
    Backtest strategy with realistic execution and costs.

    CRITICAL TRADING FLOW:
    1. At Close[t]: Generate signal, compute target weights W[t]
    2. At Open[t+1]: Execute trades, incur costs
    3. At Open[t+2]: Value portfolio, realize P&L

    ALIGNMENT LOGIC:
    - Weights generated at Close[t] are executed at Open[t+1]
    - Returns from Open[t+1] to Open[t+2] apply to weights[t]
    - This requires careful index shifting to avoid look-ahead bias

    Parameters:
    -----------
    weights_df : pd.DataFrame
        Target weights indexed by signal date (Close date)
    open_returns_df : pd.DataFrame
        Open-to-Open returns for P&L calculation
    close_prices_df : pd.DataFrame
        Close prices (for reference)
    open_prices_df : pd.DataFrame
        Open prices (for execution timing)
    transaction_cost_bps : float
        Round-trip transaction costs in basis points
    slippage_bps : float
        Execution slippage in basis points

    Returns:
    --------
    dict with backtest results including equity curve, metrics, costs
    """
    print("\n" + "=" * 60)
    print("BACKTESTING WITH REALISTIC EXECUTION")
    print("=" * 60)
    weights_shifted = weights_df.shift(1)
    common_dates = weights_shifted.index.intersection(open_returns_df.index)
    w = weights_shifted.loc[common_dates]
    r = open_returns_df.loc[common_dates]
    valid_idx = w.notna().all(axis=1) & r.notna().all(axis=1)
    w = w[valid_idx]
    r = r[valid_idx]
    print(f"\nBacktest period: {w.index[0]} to {w.index[-1]}")
    print(f"Number of trading days: {len(w)}")
    gross_returns = (w * r).sum(axis=1)
    turnover = calculate_turnover(w)
    total_cost_bps = transaction_cost_bps + slippage_bps
    transaction_costs = turnover * (total_cost_bps / 10000)
    net_returns = gross_returns - transaction_costs
    mean_gross = gross_returns.mean() * freq
    vol_gross = gross_returns.std(ddof=1) * np.sqrt(freq)
    sharpe_gross = (mean_gross - rf) / (vol_gross + 1e-12)
    mean_net = net_returns.mean() * freq
    vol_net = net_returns.std(ddof=1) * np.sqrt(freq)
    sharpe_net = (mean_net - rf) / (vol_net + 1e-12)
    gross_curve = (1 + gross_returns).cumprod()
    net_curve = (1 + net_returns).cumprod()
    running_max_gross = gross_curve.expanding().max()
    drawdown_gross = (gross_curve - running_max_gross) / running_max_gross
    max_dd_gross = drawdown_gross.min()
    running_max_net = net_curve.expanding().max()
    drawdown_net = (net_curve - running_max_net) / running_max_net
    max_dd_net = drawdown_net.min()
    total_turnover = turnover.sum()
    avg_daily_turnover = turnover.mean()
    total_costs = transaction_costs.sum()
    cost_impact = mean_gross - mean_net
    print(f"\n{'STRATEGY PERFORMANCE':^60}")
    print("-" * 60)
    print(f"Gross Annual Return:    {mean_gross:>8.2%}")
    print(f"Net Annual Return:      {mean_net:>8.2%}")
    print(f"Annual Volatility:      {vol_net:>8.2%}")
    print(f"Sharpe Ratio (Net):     {sharpe_net:>8.2f}")
    print(f"Max Drawdown (Net):     {max_dd_net:>8.2%}")
    print(f"\n{'COST ANALYSIS':^60}")
    print("-" * 60)
    print(f"Total Turnover:         {total_turnover:>8.2f}x")
    print(f"Avg Daily Turnover:     {avg_daily_turnover:>8.2%}")
    print(f"Cost Drag (Annual):     {cost_impact:>8.2%}")
    print(f"Transaction Cost (bps): {transaction_cost_bps:>8.1f}")
    print(f"Slippage (bps):         {slippage_bps:>8.1f}")
    return {
        "gross_returns": gross_returns,
        "net_returns": net_returns,
        "gross_curve": gross_curve,
        "net_curve": net_curve,
        "drawdown_gross": drawdown_gross,
        "drawdown_net": drawdown_net,
        "weights": w,
        "turnover": turnover,
        "transaction_costs": transaction_costs,
        "metrics": {
            "annual_return_gross": float(mean_gross),
            "annual_return_net": float(mean_net),
            "annual_vol_gross": float(vol_gross),
            "annual_vol_net": float(vol_net),
            "sharpe_gross": float(sharpe_gross),
            "sharpe_net": float(sharpe_net),
            "max_drawdown_gross": float(max_dd_gross),
            "max_drawdown_net": float(max_dd_net),
            "total_turnover": float(total_turnover),
            "avg_daily_turnover": float(avg_daily_turnover),
            "cost_drag": float(cost_impact),
        },
    }


def benchmark_backtest(benchmark_returns):
    """
    Calculate benchmark (SPY) performance metrics.

    Simple buy-and-hold strategy with no rebalancing costs.
    """
    mean_ret = benchmark_returns.mean() * ANNUALIZATION_FACTOR
    vol = benchmark_returns.std(ddof=1) * np.sqrt(ANNUALIZATION_FACTOR)
    sharpe = (mean_ret - RISK_FREE_RATE) / (vol + 1e-12)
    cum_ret = (1 + benchmark_returns).cumprod()
    running_max = cum_ret.expanding().max()
    drawdown = (cum_ret - running_max) / running_max
    max_dd = drawdown.min()
    return {
        "returns": benchmark_returns,
        "curve": cum_ret,
        "drawdown": drawdown,
        "metrics": {
            "annual_return": float(mean_ret),
            "annual_vol": float(vol),
            "sharpe": float(sharpe),
            "max_drawdown": float(max_dd),
        },
    }


def plot_results(strategy_results, benchmark_results, start_date, end_date):
    """
    Create comprehensive visualization of strategy vs benchmark.
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle(
        f"DCC-GARCH Strategy Performance vs {BENCHMARK_TICKER}\n{start_date} to {end_date}",
        fontsize=14,
        fontweight="bold",
    )
    ax1 = axes[0]
    strategy_results["net_curve"].plot(
        ax=ax1, label="DCC-GARCH (Net)", linewidth=2, color="darkblue"
    )
    strategy_results["gross_curve"].plot(
        ax=ax1,
        label="DCC-GARCH (Gross)",
        linewidth=1.5,
        linestyle="--",
        color="lightblue",
        alpha=0.7,
    )
    benchmark_results["curve"].plot(ax=ax1, label=BENCHMARK_TICKER, linewidth=2, color="red")
    ax1.set_ylabel("Cumulative Return", fontsize=11)
    ax1.set_title("Equity Curves", fontsize=12, fontweight="bold")
    ax1.legend(loc="best")
    ax1.grid(True, alpha=0.3)
    ax2 = axes[1]
    strategy_results["drawdown_net"].plot(
        ax=ax2, label="DCC-GARCH", linewidth=2, color="darkblue", alpha=0.7
    )
    benchmark_results["drawdown"].plot(
        ax=ax2, label=BENCHMARK_TICKER, linewidth=2, color="red", alpha=0.7
    )
    ax2.fill_between(
        strategy_results["drawdown_net"].index,
        strategy_results["drawdown_net"].values,
        0,
        alpha=0.3,
        color="darkblue",
    )
    ax2.fill_between(
        benchmark_results["drawdown"].index,
        benchmark_results["drawdown"].values,
        0,
        alpha=0.3,
        color="red",
    )
    ax2.set_ylabel("Drawdown", fontsize=11)
    ax2.set_title("Underwater Curves", fontsize=12, fontweight="bold")
    ax2.legend(loc="best")
    ax2.grid(True, alpha=0.3)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax3 = axes[2]
    window = min(252, len(strategy_results["net_returns"]) // 2)
    if window >= 60:
        strat_rolling_sharpe = (
            strategy_results["net_returns"].rolling(window).mean()
            * 252
            / (strategy_results["net_returns"].rolling(window).std() * np.sqrt(252))
        )
        bench_rolling_sharpe = (
            benchmark_results["returns"].rolling(window).mean()
            * 252
            / (benchmark_results["returns"].rolling(window).std() * np.sqrt(252))
        )
        strat_rolling_sharpe.plot(ax=ax3, label="DCC-GARCH", linewidth=2, color="darkblue")
        bench_rolling_sharpe.plot(ax=ax3, label=BENCHMARK_TICKER, linewidth=2, color="red")
        ax3.axhline(y=0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
        ax3.set_ylabel("Sharpe Ratio", fontsize=11)
        ax3.set_title(f"Rolling Sharpe Ratio ({window}-day window)", fontsize=12, fontweight="bold")
        ax3.legend(loc="best")
        ax3.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"\n{'PERFORMANCE COMPARISON':^60}")
    print("=" * 60)
    print(f"{'Metric':<30} {'DCC-GARCH':>14} {BENCHMARK_TICKER:>14}")
    print("-" * 60)
    strat = strategy_results["metrics"]
    bench = benchmark_results["metrics"]
    print(
        f"{'Annual Return':<30} {strat['annual_return_net']:>13.2%} {bench['annual_return']:>13.2%}"
    )
    print(
        f"{'Annual Volatility':<30} {strat['annual_vol_net']:>13.2%} {bench['annual_vol']:>13.2%}"
    )
    print(f"{'Sharpe Ratio':<30} {strat['sharpe_net']:>13.2f} {bench['sharpe']:>13.2f}")
    print(f"{'Max Drawdown':<30} {strat['max_drawdown_net']:>13.2%} {bench['max_drawdown']:>13.2%}")
    print(f"{'Cost Drag':<30} {strat['cost_drag']:>13.2%} {'N/A':>14}")
    print("=" * 60)
    outperf = strat["annual_return_net"] - bench["annual_return"]
    print(f"\nOutperformance: {outperf:+.2%} per annum")


def run_production_strategy(
    tickers=MAG7_TICKERS, benchmark=BENCHMARK_TICKER, start=BACKTEST_START, end=BACKTEST_END
):
    """
    End-to-end execution of DCC-GARCH trading strategy.

    PIPELINE:
    1. Fetch data from Yahoo Finance
    2. Calculate Close-to-Close returns for modeling
    3. Fit univariate GARCH(1,1) models
    4. Estimate DCC parameters
    5. Filter time-varying covariances
    6. Compute minimum variance weights
    7. Backtest with realistic execution
    8. Compare with benchmark
    9. Visualize results
    """
    print("=" * 60)
    print("DCC-GARCH PRODUCTION STRATEGY FRAMEWORK")
    print("=" * 60)
    print(f"Universe: {', '.join(tickers)}")
    print(f"Benchmark: {benchmark}")
    print(f"Period: {start} to {end}")
    print(f"Transaction Cost: {TRANSACTION_COST_BPS} bps")
    print(f"Slippage: {SLIPPAGE_BPS} bps")
    print("=" * 60)
    data = load_portfolio_data(tickers, start, end)
    benchmark_data = fetch_benchmark_data(benchmark, start, end)
    open_prices = data["Open"]
    close_prices = data["Close"]
    close_returns = calculate_close_to_close_returns(close_prices, freq=ANNUALIZATION_FACTOR)
    open_returns = calculate_open_to_open_returns(open_prices)
    benchmark_open_returns = calculate_open_to_open_returns(benchmark_data[["Open"]])
    common_dates = open_returns.index.intersection(benchmark_open_returns.index)
    benchmark_open_returns = benchmark_open_returns.loc[common_dates].squeeze()
    garch_models, cond_vol, z = fit_univariate_garch(
        close_returns, p=GARCH_P, q=GARCH_Q, dist=GARCH_DISTRIBUTION, mean=GARCH_MEAN_MODEL
    )
    dcc_params = fit_dcc(z, a0=DCC_INITIAL_A, b0=DCC_INITIAL_B)
    if not dcc_params["success"]:
        print("\nWARNING: DCC optimization did not converge. Results may be unreliable.")
    R_t, Sigma_t = dcc_filter(z, cond_vol, dcc_params["a"], dcc_params["b"])
    weights = min_var_weights(Sigma_t)
    strategy_results = backtest_strategy(
        weights,
        open_returns,
        close_prices,
        open_prices,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        slippage_bps=SLIPPAGE_BPS,
        freq=ANNUALIZATION_FACTOR,
        rf=RISK_FREE_RATE,
    )
    benchmark_results = benchmark_backtest(benchmark_open_returns)
    print(f"\n{'BENCHMARK PERFORMANCE':^60}")
    print("-" * 60)
    print(f"Annual Return:          {benchmark_results['metrics']['annual_return']:>8.2%}")
    print(f"Annual Volatility:      {benchmark_results['metrics']['annual_vol']:>8.2%}")
    print(f"Sharpe Ratio:           {benchmark_results['metrics']['sharpe']:>8.2f}")
    print(f"Max Drawdown:           {benchmark_results['metrics']['max_drawdown']:>8.2%}")
    plot_results(strategy_results, benchmark_results, start, end)
    return {
        "data": data,
        "close_returns": close_returns,
        "open_returns": open_returns,
        "garch_models": garch_models,
        "cond_vol": cond_vol,
        "standardized_residuals": z,
        "dcc_params": dcc_params,
        "correlations": R_t,
        "covariances": Sigma_t,
        "weights": weights,
        "strategy_results": strategy_results,
        "benchmark_results": benchmark_results,
    }


if __name__ == "__main__":
    results = run_production_strategy(
        tickers=MAG7_TICKERS, benchmark=BENCHMARK_TICKER, start=BACKTEST_START, end=BACKTEST_END
    )
    print("\n" + "=" * 60)
    print("EXECUTION COMPLETE")
    print("=" * 60)
    print("\nStrategy ready for production deployment.")
    print("All results stored in 'results' dictionary.")
